# Frozen wav2vec 2.0 spoof detection
Run 01, then 02 or 03. Each notebook uses a fresh kernel. Smoke mode and validation-only evaluation are defaults. Outputs are local and ignored. Final test evaluation is explicitly opt-in; external evaluation is deferred.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/datasets.py').exists())
sys.path.insert(0, str(project_dir))
from src.datasets import dataset_path
from src.audio.features import SETTINGS, mfcc_features
from src.audio.workflow import prepare_frames, fit_export

FULL_RUN = False
FINAL_EVALUATION = False  # Opt in only after model choices are frozen.
output_dir = project_dir / 'outputs/audio' / ('full' if FULL_RUN else 'smoke') / ('final' if FINAL_EVALUATION else 'development')
la_dir = dataset_path('asvspoof19_la')
frames = prepare_frames(la_dir, output_dir, full=FULL_RUN, final=FINAL_EVALUATION)
display(pd.DataFrame({s: f['label'].value_counts() for s, f in frames.items()}))


In [ ]:
import torch
from src.audio.encoder import load_encoder, encoder_features, cache_path

baseline_metrics = pd.read_csv(output_dir / 'mfcc_metrics.csv')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor, encoder = load_encoder(device)
features = {split: encoder_features(frame, processor, encoder, device, cache=cache_path(output_dir, split)) for split, frame in frames.items()}
model, threshold, metrics = fit_export('wav2vec2', features, frames, output_dir)
display(pd.concat([baseline_metrics, metrics], ignore_index=True))
